# 香氛识别

1. 信号进样点检测 + 40s对齐 + 导出xlsx（原始矩阵格式）
2. 对齐之后减掉基准值将所有通道拉至同一起点看变化值
2. 绘制四季4张图，且每张图绘制该季节全部样本的全部30通道，共计30*样本数条曲线

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 11

# 固定输出到 Fragrance/result（无论你从哪个工作目录启动Jupyter）。
cwd = Path.cwd()
if (cwd / 'Fragrance' / 'data').exists():
    PROJECT_DIR = cwd / 'Fragrance'
elif (cwd / 'data').exists() and (cwd / 'test.ipynb').exists():
    PROJECT_DIR = cwd
else:
    raise FileNotFoundError('未找到 Fragrance/data，请在项目根目录或 Fragrance 目录启动 notebook。')

DATA_DIR = PROJECT_DIR / 'data'
RESULT_DIR = PROJECT_DIR / 'result'
XLSX_DIR = RESULT_DIR / 'xlsx'
FIG_DIR = RESULT_DIR / 'fig1_split'

XLSX_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

LABEL_DICT = {'chun': 0, 'xia': 1, 'qiu': 2, 'dong': 3}
CLASS_NAMES = ['春季香氛', '夏季香氛', '秋季香氛', '冬季香氛']
CLASS_KEY = ['chun', 'xia', 'qiu', 'dong']

print(f'项目目录: {PROJECT_DIR}')
print(f'数据目录: {DATA_DIR}')
print(f'xlsx输出目录: {XLSX_DIR}')
print(f'图像输出目录: {FIG_DIR}')

项目目录: e:\桌面\香氛-陈莹\Fragrance
数据目录: e:\桌面\香氛-陈莹\Fragrance\data
xlsx输出目录: e:\桌面\香氛-陈莹\Fragrance\result\xlsx
图像输出目录: e:\桌面\香氛-陈莹\Fragrance\result\fig1_split


In [17]:
def detect_injection_point(signal, search_start=35, search_end=55):
    sig = savgol_filter(signal, 11, 2, axis=0)
    diff = np.abs(np.diff(sig, axis=0).mean(axis=1))
    search_zone = diff[search_start:search_end]
    if len(search_zone) == 0:
        return 40
    return int(np.argmax(search_zone) + search_start)


def align_to_40s(signal, real_inj_idx, align_target=40):
    """时间平移到40s后，按40s值做通道归一基准，使变化点对齐到同一起点。"""
    n_rows, n_cols = signal.shape
    shift = align_target - real_inj_idx

    # 空位保留为NaN，确保原始有效点只发生行位移。
    aligned = np.full((n_rows, n_cols), np.nan, dtype=float)

    if shift >= 0:
        src_start = 0
        src_end = n_rows - shift
        dst_start = shift
        dst_end = shift + (src_end - src_start)
    else:
        src_start = -shift
        src_end = n_rows
        dst_start = 0
        dst_end = src_end - src_start

    if src_end > src_start:
        aligned[dst_start:dst_end, :] = signal[src_start:src_end, :]

    # 以40s时刻为锚点，强制各通道在变化起点对齐。
    if 0 <= align_target < n_rows:
        anchor = aligned[align_target, :]
    else:
        anchor = np.full((n_cols,), np.nan)

    if np.isnan(anchor).all():
        valid_rows = np.where(~np.isnan(aligned).all(axis=1))[0]
        if len(valid_rows) > 0:
            anchor = aligned[valid_rows[0], :]

    aligned = aligned - anchor
    return aligned

In [18]:
def load_align_and_export(data_dir=DATA_DIR, out_dir=XLSX_DIR):
    file_list = sorted([p for p in data_dir.iterdir() if p.is_file() and p.suffix.lower() in [".xlsx", ".xls"]])
    print(f"读取文件数量: {len(file_list)}")

    X_aligned, y, files = [], [], []
    summary_rows = []

    def infer_label_from_name(fpath):
        name = fpath.stem.lower()
        label_keywords = {
            0: ["chun", "春"],
            1: ["xia", "夏"],
            2: ["qiu", "秋"],
            3: ["dong", "冬"],
        }
        for label, keys in label_keywords.items():
            if any(k in name for k in keys):
                return label
        return None

    for fpath in file_list:
        try:
            raw = pd.read_excel(fpath, header=None).values.astype(float)
        except Exception as e:
            print(f"[跳过] {fpath.name} 读取失败: {e}")
            continue

        label = infer_label_from_name(fpath)
        if label is None:
            print(f"[跳过] {fpath.name} 未匹配到类别（按文件名中的春夏秋冬/chunxiaqiudong关键词）")
            continue

        inj_idx = detect_injection_point(raw, 35, 55)
        aligned = align_to_40s(raw, inj_idx, align_target=40)

        X_aligned.append(aligned)   # 保持 list，不转 np.array
        y.append(label)
        files.append(fpath.name)

        rel_stem = "_".join(fpath.relative_to(data_dir).parts)
        rel_stem = rel_stem.rsplit('.', 1)[0]
        out_name = f"{rel_stem}_aligned40.xlsx"
        out_path = out_dir / out_name
        pd.DataFrame(aligned).to_excel(out_path, index=False, header=False)

        summary_rows.append({
            "source_file": fpath.name,
            "class_name": CLASS_NAMES[label],
            "detected_injection_index": inj_idx,
            "output_xlsx": out_name,
            "rows": aligned.shape[0],
            "cols": aligned.shape[1],
        })

    if len(X_aligned) == 0:
        raise ValueError("无有效数据，未生成任何对齐结果。")

    y = np.array(y)
    pd.DataFrame(summary_rows).to_excel(out_dir / "alignment_summary.xlsx", index=False)

    print(f"完成对齐样本: {len(X_aligned)}")
    n_channels = X_aligned[0].shape[1]
    print(f"通道数: {n_channels}")
    for i, cname in enumerate(CLASS_NAMES):
        n = int((y == i).sum())
        print(f"{cname}: 样本 {n}，应绘曲线 {n * n_channels}")

    return X_aligned, y, files

X_data, y, files = load_align_and_export()


读取文件数量: 13
[跳过] ~$chun22.xlsx 读取失败: [Errno 13] Permission denied: 'e:\\桌面\\香氛-陈莹\\Fragrance\\data\\~$chun22.xlsx'
完成对齐样本: 12
通道数: 30
春季香氛: 样本 3，应绘曲线 90
夏季香氛: 样本 3，应绘曲线 90
秋季香氛: 样本 3，应绘曲线 90
冬季香氛: 样本 3，应绘曲线 90


In [19]:
def plot_fig1_split_all_channels(X_data, y, out_dir=FIG_DIR):
    colors = plt.cm.tab10.colors

    for i, class_name in enumerate(CLASS_NAMES):
        subset = [x for x, label in zip(X_data, y) if label == i]
        if len(subset) == 0:
            print(f'[跳过] {class_name} 无样本')
            continue

        n_samples = len(subset)
        n_channels = subset[0].shape[1]
        fig, ax = plt.subplots(figsize=(12, 8))

        # 仅绘制：样本数 * 通道数 条曲线
        for sample in subset:
            for ch in range(n_channels):
                ax.plot(sample[:, ch], color=colors[i], alpha=0.7, linewidth=0.5)

        ax.axvline(x=40, color='gray', linestyle=':', alpha=0.9, linewidth=1.2)
        ax.text(41, ax.get_ylim()[1] * 0.95, '进样点 40s', color='gray', fontsize=10)

        # 按要求：标题不显示样本数量
        ax.set_title(f'{class_name} 响应曲线', fontsize=14, fontweight='bold')
        ax.set_xlabel('时间 (s)')
        ax.set_ylabel('响应值 (V)')
        ax.grid(alpha=0.25, linestyle=':')

        out_name = f'fig1_{CLASS_KEY[i]}_all_channels.png'
        out_path = out_dir / out_name
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

        print(f'{class_name}: 实际绘制曲线 {n_samples * n_channels} 条 -> {out_path}')


plot_fig1_split_all_channels(X_data, y)

春季香氛: 实际绘制曲线 90 条 -> e:\桌面\香氛-陈莹\Fragrance\result\fig1_split\fig1_chun_all_channels.png
夏季香氛: 实际绘制曲线 90 条 -> e:\桌面\香氛-陈莹\Fragrance\result\fig1_split\fig1_xia_all_channels.png
秋季香氛: 实际绘制曲线 90 条 -> e:\桌面\香氛-陈莹\Fragrance\result\fig1_split\fig1_qiu_all_channels.png
冬季香氛: 实际绘制曲线 90 条 -> e:\桌面\香氛-陈莹\Fragrance\result\fig1_split\fig1_dong_all_channels.png


In [23]:
print('输出完成')

输出完成
